# Step 2.3 — YOLOv5 Camera Detection (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_1/camera_meta.json` (from Step 1.1) |
| **Outputs** | `output/step_2/yolo/<sample>/<camera>.json` — 2D bounding boxes |
| | `output/step_2/yolo_empty_detection_report.csv` — diagnostic breakdown |
| **Used by** | Step 2.3.1 (global 3D projection — see next notebook), Step 4.3 (camera tracking) |

---

### Fixed
- Path now points to `STEP1_DIR / "camera_meta.json"` directly (no emoji subfolder — matches the upgraded Step 1.1 output location).
- Added a diagnostic that breaks down your 382 empty-detection files **by camera channel**. If they cluster on back-facing cameras, that's mostly genuine empty scenes. If spread evenly, it points to the model's recall limit.

In [1]:
#pip install ultralytics

In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import DATA_ROOT, STEP1_DIR, STEP2_DIR

YOLO_OUT_DIR = STEP2_DIR / "yolo"
YOLO_OUT_DIR.mkdir(parents=True, exist_ok=True)

cam_meta_path = STEP1_DIR / "camera_meta.json"
if not cam_meta_path.exists():
    raise FileNotFoundError(f"camera_meta.json not found at {cam_meta_path} — run Step 1.1 first.")

print(f"✅ cam_meta_path : {cam_meta_path}")
print(f"✅ YOLO_OUT_DIR  : {YOLO_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ cam_meta_path : F:\Sensor fusion Research\output\step_1\camera_meta.json
✅ YOLO_OUT_DIR  : F:\Sensor fusion Research\output\step_2\yolo


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Run YOLOv5 on all camera images
# ─────────────────────────────────────────────────────────────────

import torch, json, cv2
from tqdm import tqdm

with open(cam_meta_path) as f:
    cam_meta_data = json.load(f)

# NOTE: to upgrade to YOLOv8 instead, replace this block with:
#   from ultralytics import YOLO
#   model = YOLO('yolov8n.pt')
# and change the inference call in the loop below to: results = model(img_rgb, conf=0.25)
# ------->model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.conf = 0.25
model.iou = 0.45
model.classes = None

total_detections = 0
empty_files = []

#for sample_id, cam_views in tqdm(cam_meta_data.items(), desc="Running YOLOv5"):
for sample_id, cam_views in tqdm(cam_meta_data.items(), desc="Running YOLOv8"):
    for cam_name, entry in cam_views.items():
        try:
            img_path = DATA_ROOT / entry["filename"]
            if not img_path.exists():
                print(f"❌ Image not found: {img_path}")
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                print(f"❌ Failed to load: {img_path}")
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            #results = model(img_rgb, size=640)

            results = model(img_rgb, imgsz=640, conf=0.25)
            
            det_list = []
            
            for r in results:
                boxes = r.boxes
            
                for b in boxes:
                    det_list.append({
                        "xmin": int(b.xyxy[0][0].item()),
                        "ymin": int(b.xyxy[0][1].item()),
                        "xmax": int(b.xyxy[0][2].item()),
                        "ymax": int(b.xyxy[0][3].item()),
                        "confidence": float(b.conf[0].item()),
                        "class_id": int(b.cls[0].item()),
                        "class_name": model.names[int(b.cls[0].item())]
                    })
            
            #preds = results.pandas().xyxy[0]
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            results = model(img_rgb, imgsz=640, conf=0.25)
            
            det_list = []
            
            for r in results:
                for b in r.boxes:
                    det_list.append({
                        "xmin": int(b.xyxy[0][0].item()),
                        "ymin": int(b.xyxy[0][1].item()),
                        "xmax": int(b.xyxy[0][2].item()),
                        "ymax": int(b.xyxy[0][3].item()),
                        "confidence": float(b.conf[0].item()),
                        "class_id": int(b.cls[0].item()),
                        "class_name": model.names[int(b.cls[0].item())]
                    })
            
            out_dir = YOLO_OUT_DIR / sample_id
            out_dir.mkdir(parents=True, exist_ok=True)
            
            out_file = out_dir / f"{cam_name}.json"
            
            with open(out_file, "w") as f:
                json.dump(det_list, f, indent=2)
            
            total_detections += len(det_list)
            
            if len(det_list) == 0:
                empty_files.append({
                    "sample_id": sample_id,
                    "camera": cam_name
                })

           # det_list = []
            #for _, row in preds.iterrows():
                #det_list.append({
                    #'xmin': int(row['xmin']), 'ymin': int(row['ymin']),
                    #'xmax': int(row['xmax']), 'ymax': int(row['ymax']),
                    #'confidence': float(row['confidence']),
                    #'class_id': int(row['class']),
                    #'class_name': row['name']
                #})

            out_dir = YOLO_OUT_DIR / sample_id
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / f"{cam_name}.json"
            with open(out_file, 'w') as f:
                json.dump(det_list, f, indent=2)

            total_detections += len(det_list)
            if len(det_list) == 0:
                empty_files.append({"sample_id": sample_id, "camera": cam_name})

        except KeyError as e:
            print(f"❌ Missing key in metadata: {e}")
        except Exception as e:
            print(f"❌ Unexpected error for {entry}: {e}")

print(f"\n✅ All YOLO detections saved to: {YOLO_OUT_DIR}")
print(f"   Total detections: {total_detections}")
print(f"   Empty files: {len(empty_files)}")

Running YOLOv8:   0%|                                                                          | 0/404 [00:00<?, ?it/s]


0: 384x640 4 cars, 1 truck, 318.7ms
Speed: 37.8ms preprocess, 318.7ms inference, 48.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 truck, 65.2ms
Speed: 4.1ms preprocess, 65.2ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 boat, 42.9ms
Speed: 2.7ms preprocess, 42.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 boat, 57.5ms
Speed: 1.9ms preprocess, 57.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 65.8ms
Speed: 1.9ms preprocess, 65.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 47.8ms
Speed: 3.4ms preprocess, 47.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 2 cars, 51.9ms
Speed: 1.8ms preprocess, 51.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 2 cars, 39.2ms
Speed: 2.2ms preprocess, 39.2ms inference, 0.7ms postpr

Running YOLOv8:   0%|▏                                                                 | 1/404 [00:05<36:08,  5.38s/it]


0: 384x640 1 person, 4 cars, 1 bus, 1 truck, 46.8ms
Speed: 2.5ms preprocess, 46.8ms inference, 5.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 4 cars, 1 bus, 1 truck, 55.7ms
Speed: 3.2ms preprocess, 55.7ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 47.7ms
Speed: 2.3ms preprocess, 47.7ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 49.7ms
Speed: 5.6ms preprocess, 49.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 bench, 42.5ms
Speed: 2.6ms preprocess, 42.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 bench, 57.5ms
Speed: 3.3ms preprocess, 57.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 2 cars, 39.9ms
Speed: 3.0ms preprocess, 39.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 

Running YOLOv8:   0%|▎                                                                 | 2/404 [00:06<18:36,  2.78s/it]

...


0: 384x640 1 car, 36.5ms
Speed: 1.8ms preprocess, 36.5ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 38.8ms
Speed: 2.0ms preprocess, 38.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 bus, 40.0ms
Speed: 2.0ms preprocess, 40.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 bus, 38.1ms
Speed: 1.9ms preprocess, 38.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 43.5ms
Speed: 3.6ms preprocess, 43.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 39.7ms
Speed: 2.2ms preprocess, 39.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 1 car, 1 traffic light, 35.7ms
Speed: 1.8ms preprocess, 35.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 1 car, 1 traffic light, 36.0ms
Speed: 1.6ms preprocess, 36.0ms inference

Running YOLOv8: 100%|███████████████████████████████████████████████████████████████▊| 403/404 [05:28<00:00,  1.24it/s]


0: 384x640 4 cars, 38.6ms
Speed: 1.9ms preprocess, 38.6ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 69.1ms
Speed: 1.6ms preprocess, 69.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 36.6ms
Speed: 1.9ms preprocess, 36.6ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 37.1ms
Speed: 1.9ms preprocess, 37.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 34.9ms
Speed: 1.5ms preprocess, 34.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 59.4ms
Speed: 1.7ms preprocess, 59.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 1 car, 2 traffic lights, 39.3ms
Speed: 1.8ms preprocess, 39.3ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 1 car, 2 traffic lights, 39.4ms
Speed: 2.1ms preprocess, 39.4ms inferen

Running YOLOv8: 100%|████████████████████████████████████████████████████████████████| 404/404 [05:29<00:00,  1.23it/s]


✅ All YOLO detections saved to: F:\Sensor fusion Research\output\step_2\yolo
   Total detections: 18198
   Empty files: 978


## Diagnostic: WHY are detections empty? Breakdown by camera channel

In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Break down empty detections by camera channel
# If BACK cameras dominate -> mostly genuine empty road scenes
# If spread evenly -> points to model recall limit (worth upgrading model size)
# ─────────────────────────────────────────────────────────────────

import pandas as pd

empty_df = pd.DataFrame(empty_files)

if len(empty_df) > 0:
    breakdown = empty_df["camera"].value_counts().reset_index()
    breakdown.columns = ["camera", "empty_count"]
    breakdown["pct_of_empties"] = (breakdown["empty_count"] / len(empty_df) * 100).round(1)

    print(f"Total empty detection files: {len(empty_df)}\n")
    display(breakdown)

    back_cams = {"CAM_BACK", "CAM_BACK_LEFT", "CAM_BACK_RIGHT"}
    back_pct = breakdown[breakdown["camera"].isin(back_cams)]["pct_of_empties"].sum()

    print(f"\n{'👉 ' if back_pct > 60 else '⚠️ '}{back_pct:.0f}% of empty detections are on back-facing cameras.")
    if back_pct > 60:
        print("   This is mostly consistent with genuinely empty rear scenes — not a bug.")
    else:
        print("   Empties are spread across front and back cameras — this points to the")
        print("   YOLOv5s model's recall limit rather than empty scenes. Consider upgrading")
        print("   to yolov5m/yolov5l, or YOLOv8, for better small/distant-object recall.")

    report_path = STEP2_DIR / "yolo_empty_detection_report.csv"
    breakdown.to_csv(report_path, index=False)
    print(f"\n📄 Saved: {report_path}")
else:
    print("✅ No empty detection files.")

Total empty detection files: 978



,camera,empty_count,pct_of_empties
0,CAM_BACK_LEFT,338,34.6
1,CAM_BACK_RIGHT,296,30.3
2,CAM_FRONT_LEFT,216,22.1
3,CAM_FRONT_RIGHT,74,7.6
4,CAM_BACK,44,4.5
5,CAM_FRONT,10,1.0



👉 69% of empty detections are on back-facing cameras.
   This is mostly consistent with genuinely empty rear scenes — not a bug.

📄 Saved: F:\Sensor fusion Research\output\step_2\yolo_empty_detection_report.csv
